# E34 — Generator-Robust Audio–Text Alignment with Group DRO

This notebook builds on E33 and tests whether generator-aware robust optimization improves transfer to unseen audio generators. The same M2D-CLAP backbone, custom LoRA injection, audio/text encoding, interaction features, ordinal head, and E33 ranking losses are retained. No challenge performance is assumed or fabricated.

## What changed from E33

- Added `erm`, `balanced_erm`, `group_dro`, and `balanced_group_dro` training modes. Balanced sampling changes the effective training distribution; Group DRO changes the objective while leaving the default sampler shuffled.
- Group DRO uses per-example CORAL plus weighted Huber losses, averages them within each group present in a minibatch, and updates persistent log-space group weights. Pairwise and soft-Spearman objectives remain batch-level auxiliaries.
- Treats the observed `audiocaps` group as a real-audio domain rather than a synthetic generator. It can contribute as its own training domain but is excluded from synthetic-generator LOGO folds.
- Implements fold-local label normalization and nested training-side validation for LOGO. A held-out generator is not used for training, early stopping, or model selection.
- All paths are user-configurable. Training is gated until dataset and checkpoint paths are configured.

## Run on Colab

1. Open this notebook in Colab with a T4 GPU runtime.
2. Configure the three Drive paths in the next code cell. `drive_data_path` must point to the supplied dataset directory; `drive_checkpoint_path` to the official E33 M2D-CLAP `checkpoint-30.pth`; and `drive_output_path` to a writable Drive output directory.
3. Run all cells. With `experiment="group_dro"`, E0 ERM and E2 Group DRO run under the same seed and settings. Set `experiment="comparison"` to include balanced ERM and balanced Group DRO as well.
4. Set `run_logo=True` only when ready for the much longer leave-one-generator-out evaluation. The official hidden/test split is not used for model selection.

The dataset and checkpoint locations are intentionally blank. No Drive IDs or training results are embedded.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install nnAudio einops timm transformers sentence-transformers scipy pandas soundfile matplotlib


Mounted at /content/drive


In [ ]:

from pathlib import Path
import subprocess
import zipfile
import shutil

CKPT_ROOT = Path("/content/m2d_ckpt")
CKPT_DIR = CKPT_ROOT / "m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025"
CKPT = CKPT_DIR / "checkpoint-30.pth"

CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not CKPT.exists():
    print("Downloading M2D-CLAP checkpoint...")

    url = (
        "https://github.com/nttcslab/m2d/releases/download/v0.5.0/"
        "m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip"
    )

    zip_path = CKPT_ROOT / "ckpt.zip"

    subprocess.run(
        ["wget", "-O", str(zip_path), url],
        check=True
    )

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(CKPT_ROOT)

    if not CKPT.exists():
        found = list(CKPT_ROOT.rglob("checkpoint-30.pth"))
        if not found:
            raise FileNotFoundError(
                "Checkpoint not found after extraction!"
            )
        shutil.copy2(found[0], CKPT)

    zip_path.unlink(missing_ok=True)

print("Checkpoint:", CKPT)
print("Exists:", CKPT.exists())
print("Size:", round(CKPT.stat().st_size / 1e6), "MB")


Checkpoint: /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth
Exists: True
Size: 1695 MB


## 1. Portable M2D-CLAP runtime

This is the E33 portable runtime cell, retained so the notebook can load the same M2D-CLAP checkpoint without replacing the backbone.


In [ ]:
"""Masked Modeling Duo (M2D) Portable Runtime.

Basic requirements:
    pip install timm einops nnAudio transformers

Additional requirements for CLAP:
    pip install sentence_transformers
"""

import logging
import os
import numpy as np
from pathlib import Path
from functools import partial
import re

import torch
import timm
from timm.layers import trunc_normal_
from einops import rearrange
import nnAudio.features


class Config:
    weight_file = ''
    feature_d = 768 * 5
    model = ''
    input_size = [80, 208]
    patch_size = [16, 16]
    sr = '16k'
    flat_features = False


def expand_size(sz):
    if isinstance(sz, int):
        return [sz, sz]
    return sz


class PatchEmbed(torch.nn.Module):
    """ 2D Image to Patch Embedding -- borrowed from https://pypi.org/project/timm/0.4.12/"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768, norm_layer=None, flatten=True):
        super().__init__()
        img_size = expand_size(img_size)
        patch_size = expand_size(patch_size)
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size[0] // patch_size[0], img_size[1] // patch_size[1])
        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.flatten = flatten

        self.proj = torch.nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else torch.nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        if self.flatten:
            x = x.flatten(2).transpose(1, 2)  # BCHW -> BNC
        x = self.norm(x)
        return x


class LocalViT(timm.models.vision_transformer.VisionTransformer):
    """ Vision Transformer for M2D Audio"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # Workaround for PatchEmbed to avoid unintended assertion failure. ex) AssertionError: Input image width (102) doesn't match model (608).
        self.patch_embed = PatchEmbed(self.patch_embed.img_size, self.patch_embed.patch_size,
                                      self.patch_embed.proj.in_channels, self.patch_embed.proj.out_channels)
        self.norm_stats = torch.nn.Parameter(torch.tensor([-7.1, 4.2]), requires_grad=False)
        # We do not use the default head
        del self.head

    def patch_size(self):
        return np.array(self.patch_embed.patch_size)

    def grid_size(self):
        # Workaround for compatibility issue (timm 0.4.5 fails with: return self.patch_embed.grid_size)
        img_size = np.array(self.patch_embed.img_size)
        patch_size = self.patch_size()
        grid_size = img_size // patch_size
        return grid_size

    def forward_encoder(self, x):
        x = self.patch_embed(x)

        # add pos embed w/o cls token
        pos_embed = self.pos_embed[:, 1:, :]
        if x.shape[1] < pos_embed.shape[1]:  # shorten pos_embed for a short input
            dims = pos_embed.shape[-1]
            fbins = self.grid_size()[0]
            frames = x.shape[1] // fbins
            pos_embed = pos_embed.reshape(1, fbins, -1, dims)[:, :, :frames, :].reshape(1, fbins*frames, dims)
        x = x + pos_embed

        # append cls token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # apply Transformer blocks
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)

        return x


class AudioSemanticProj(torch.nn.Module):
    def __init__(self, embed_dim=768, sem_depth=1, sem_heads=1, sem_mlp_ratio=1):
        super().__init__()
        self.sem_token = torch.nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.sem_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(embed_dim, sem_heads, sem_mlp_ratio, qkv_bias=True, norm_layer=torch.nn.LayerNorm)
            for i in range(sem_depth)])
        self.norm = torch.nn.LayerNorm(embed_dim)
        self.apply(self._init_weights)
        self.dont_average = True

    def _init_weights(self, m):
        if isinstance(m, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(m.weight)
            if isinstance(m, torch.nn.Linear) and m.bias is not None:
                torch.nn.init.constant_(m.bias, 0)
        elif isinstance(m, torch.nn.LayerNorm):
            torch.nn.init.constant_(m.bias, 0)
            torch.nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        # append sem token
        sem_token = self.sem_token  # + self.pos_embed[:, :1, :]
        sem_tokens = sem_token.expand(x.shape[0], -1, -1)
        x = torch.cat((sem_tokens, x), dim=1)

        # apply Transformer blocks
        for blk in self.sem_blocks:
            x = blk(x)
        x = x[:, 0, :]  # use semantic token only
        x = self.norm(x)

        return x


def set_requires_grad(model, val):
    for p in model.parameters():
        p.requires_grad = val


def parse_sizes_by_name(name):
    # Parse parameters. "m2d_vit_base-80x1001p16x16p16kpXXXpYYY" -> input size: 80x1001, patch size: 16x16, sr: 16k, extra parameters: XXX and YYY
    model_cls = name.split('-')[0]
    params = name.split('-')[1]
    params = params.split('p')
    params, extra = params[:3], params[3:]
    input_str, patch_str, sr = params[0], params[1], params[2] if len(params) > 2 else '16k'
    input_size = [int(a) for a in input_str.split('x')]
    patch_size = [int(a) for a in patch_str.split('x')]
    return input_size, patch_size, sr, model_cls, extra


def drop_non_model_weights(model, checkpoint, filename, except_for=[]):  # except_for=['running_mean', 'running_var']
    def not_an_exception(k):
        for ex in except_for:
            if ex in k: return False
        return True
    model_keys = [n for n, p in model.named_parameters()]
    new_ckpt = {}
    dropped = []
    for k in checkpoint:
        if k not in model_keys and not_an_exception(k):
            dropped.append(k)
            continue
        new_ckpt[k] = checkpoint[k]
    n_org = len(checkpoint.keys())
    n_cur = len(new_ckpt.keys())
    print(f' using {n_cur} parameters, while dropped {n_org - n_cur} out of {n_org} parameters from {Path(filename).parent/Path(filename).name}'
          if n_org > n_cur else f' using {n_cur} parameters from {Path(filename).parent/Path(filename).name}')
    print(' (included audio_proj params:', [k for k in new_ckpt.keys() if 'audio_proj' in k][:5])
    print(' (included text_proj params:', [k for k in new_ckpt.keys() if 'text_proj' in k][:5])
    print(' (dropped:', dropped[:5], ')' if len(dropped) < 5 else '...)')
    return new_ckpt


def load_evar_head_parameters(checkpoint, head_norm, head):
    # Load the weights of the task head trained in the EVAR fine-tuning.
    if 'module.head.norm.running_mean' in checkpoint:
        head_norm.load_state_dict({to_k: checkpoint[k] for to_k, k in {
            'running_mean':'module.head.norm.running_mean', 'running_var':'module.head.norm.running_var'}.items()})
        head.load_state_dict({to_k: checkpoint[k] for to_k, k in {
            'weight':'module.head.mlp.mlp.0.weight', 'bias':'module.head.mlp.mlp.0.bias'}.items()})
    else:
        print(' Not an EVAR checkpoint for loading head weights.')


def reformat_ckpt_keys(checkpoint):
    # In case: checkpoint['model']
    checkpoint = checkpoint['model'] if 'model' in checkpoint else checkpoint
    # The checkpoints saved in a EVAR fine-tuning has a prefix of "module.ar.runtime.backbone", the following removes it.
    new_ckpt = {}
    for k in checkpoint:
        new_k = k.replace('module.ar.runtime.backbone.', '')  # replace
        new_ckpt[new_k] = checkpoint[k]
    return new_ckpt


def extract_weight(checkpoint, root_name):
    # if no key starts with root_name, not applicable to the checkpoint.
    if not any(k.startswith(root_name) for k in checkpoint.keys()):
        return checkpoint
    # keep only the items starts with the root_name
    new_ckpt = {k[len(root_name):]: v for k, v in checkpoint.items() if k.startswith(root_name)}
    return new_ckpt


def add_semantic_audio_proj(sem_mode, embed_dim):
    sem_params = {
        1: {'sem_depth': 1, 'sem_heads': 1, 'sem_mlp_ratio': 1},
        2: {'sem_depth': 2, 'sem_heads': 1, 'sem_mlp_ratio': 1},
    }[sem_mode]
    audio_proj = AudioSemanticProj(embed_dim=embed_dim, **sem_params)
    return audio_proj


def make_it_CLAP(model, checkpoint):
    # Return if already a CLAP model
    if hasattr(model, 'audio_proj') or checkpoint is None: return
    # Add projectors if needed
    if 'audio_proj.0.weight' in checkpoint.keys():
        print(checkpoint['audio_proj.0.weight'].shape)
        proj_hidden_dim, embed_dim = checkpoint['audio_proj.0.weight'].shape
        model.audio_proj = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, proj_hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(proj_hidden_dim, proj_hidden_dim),
        )
    if 'audio_proj.sem_token' in checkpoint.keys():
        embed_dim = checkpoint['audio_proj.sem_token'].shape[-1]
        sem_blocks_nums = [int(k.split('.')[2]) for k in checkpoint.keys() if k.startswith('audio_proj.sem_blocks.')]
        sem_mode = max(sem_blocks_nums) + 1
        model.audio_proj = add_semantic_audio_proj(sem_mode, embed_dim)
    if 'text_proj.weight' in checkpoint.keys():
        dim = checkpoint['text_proj.weight'].shape
        model.text_proj = torch.nn.Linear(dim[1], dim[0])


def parse_clap_type(name):
    # Parse parameters. "m2d_clap_base-80x1001p16x16p16kpA" -> input size: 80x1001, patch size: 16x16, sr: 16k, extra: A
    params = str(name).split('-')[1]
    params = params.split('p')
    params, extra = params[:3], params[3:]
    if len(extra) == 0:
        return 'A'
    assert extra[0] in 'ABCLMNQR'
    text_encoder_name = {'A': 'GTE base', 'B': 'BERT base', 'C': 'CLAP-L', 'L': 'GTE large', 'M': 'BERT large', 'N': 'NV-Embed-v2',
                         'Q': 'RoBERTa large', 'R': 'RoBERTa base'}
    logging.info(f' using text encoder: {text_encoder_name[extra[0]]}')
    return extra[0]


def clap_text_enc_dim(param_extra):
    if len(param_extra) == 0:
        return 768
    return {'A': 768, 'B': 768, 'C': 1024, 'L': 1024, 'M': 1024, 'N': 4096, 'Q': 1024, 'R': 768}[param_extra[0]]


def get_backbone(args, weight_file):
    args.input_size, args.patch_size, args.sr, args.model, extra = parse_sizes_by_name(Path(weight_file).parent.name)

    # Load checkpoint.
    checkpoint = torch.load(weight_file, map_location='cpu', weights_only=False)
    checkpoint = checkpoint['model'] if 'model' in checkpoint else checkpoint
    checkpoint = extract_weight(checkpoint, 'backbone.') if checkpoint is not None else None  # convert from RuntimeM2D weights
    checkpoint = reformat_ckpt_keys(checkpoint)
    # Set normalization statistics for backward compatibility. The [-7.1, 4.2] is for 2022 models.
    if 'norm_stats' not in checkpoint:
        checkpoint['norm_stats'] = torch.tensor([-7.1, 4.2])
        print(' using default norm_stats:', checkpoint['norm_stats'])

    # Create a ViT.
    model = LocalViT(
        in_chans=1, img_size=args.input_size, patch_size=args.patch_size, embed_dim=768, depth=12, num_heads=12,
        mlp_ratio=4, norm_layer=partial(torch.nn.LayerNorm, eps=1e-6))

    # Modify the model if it should be a M2D-CLAP.
    make_it_CLAP(model, checkpoint)
    if len(extra) > 1 and extra[1] == 'TI':  # text encoder included
        args.text_encoder_weight = weight_file

    # Load weights.
    dropped = drop_non_model_weights(model, checkpoint, weight_file)
    msg = model.load_state_dict(dropped)
    print(msg); logging.info(msg)

    # Make normalization statistics for the model easy to use in the downstream task.
    args.mean, args.std = model.state_dict()['norm_stats'].to('cpu').numpy()

    model.eval()
    return model, checkpoint


def get_to_melspec(cfg):
    if cfg.sr == '16k':
        cfg.sample_rate, cfg.n_fft, cfg.window_size, cfg.hop_size = 16000, 400, 400, 160
        cfg.n_mels, cfg.f_min, cfg.f_max = 80, 50, 8000
    elif cfg.sr == '32k':
        cfg.sample_rate, cfg.n_fft, cfg.window_size, cfg.hop_size = 32000, 800, 800, 320
        cfg.n_mels, cfg.f_min, cfg.f_max = 80, 50, 16000
    else:
        assert False, f'Unknown input size: {cfg.input_size}'

    to_spec = nnAudio.features.MelSpectrogram(
        sr=cfg.sample_rate,
        n_fft=cfg.n_fft,
        win_length=cfg.window_size,
        hop_length=cfg.hop_size,
        n_mels=cfg.n_mels,
        fmin=cfg.f_min,
        fmax=cfg.f_max,
        center=True,
        power=2,
        verbose=False,
    )
    logging.info(f'Runtime MelSpectrogram({cfg.sample_rate}, {cfg.n_fft}, {cfg.window_size}, {cfg.hop_size}, '
                 + f'{cfg.n_mels}, {cfg.f_min}, {cfg.f_max}):')
    logging.info(to_spec)
    return to_spec


def get_timestamps(cfg, batch_audio, x):  # Returns timestamps in milliseconds.
    audio_len = len(batch_audio[0])
    sec = audio_len / cfg.sample_rate
    x_len = len(x[0])
    step = sec / x_len * 1000 # sec -> ms
    ts = torch.tensor([step * i for i in range(x_len)]).unsqueeze(0)
    ts = ts.repeat(len(batch_audio), 1)
    return ts


class PortableM2D(torch.nn.Module):
    def __init__(self, weight_file, cfg=Config(), num_classes=None, freeze_embed=False, flat_features=None):
        super().__init__()
        self.cfg = cfg
        self.cfg.weight_file = weight_file
        self.cfg.freeze_embed = freeze_embed
        self.cfg.flat_features = self.cfg.flat_features if flat_features is None else flat_features

        # Create backbone model.
        self.backbone, checkpoint = get_backbone(self.cfg, self.cfg.weight_file)
        # Finalize feature dimension.
        d = self.backbone.pos_embed.shape[-1]
        if num_classes is not None and 'module.head.mlp.mlp.0.weight' in checkpoint and checkpoint['module.head.mlp.mlp.0.weight'].shape[-1] == d:
            self.cfg.flat_features = True
        n_stack_feature = 1 if self.cfg.flat_features else (self.cfg.input_size[0] // self.cfg.patch_size[0])
        self.cfg.feature_d = d * n_stack_feature  # 768 if flat_features else 768*5=3840
        # Create head.
        if num_classes is not None:
            self.head_norm = torch.nn.BatchNorm1d(self.cfg.feature_d, affine=False)
            self.head = torch.nn.Linear(self.cfg.feature_d, num_classes)
            trunc_normal_(self.head.weight, std=2e-5)
            load_evar_head_parameters(checkpoint, self.head_norm, self.head)
        # Option: freeze patch embedding ([2211.09359] How to Fine-Tune Vision Models with SGD)
        if self.cfg.freeze_embed:
            set_requires_grad(self.backbone.patch_embed, False)
            logging.info(' ** Freeze patch_embed **')
            logging.info(self.backbone.patch_embed)

        logging.info(f'Model input size: {self.cfg.input_size}')
        logging.info(f'Using weights: {self.cfg.weight_file}')
        logging.info(f'Feature dimension: {self.cfg.feature_d}')
        logging.info(f'Norm stats: {self.cfg.mean}, {self.cfg.std}')

        self.to_spec = get_to_melspec(self.cfg)
        self.eval()

    def to_log_mel_spec(self, batch_audio):
        x = self.to_spec(batch_audio)
        x = (x + torch.finfo().eps).log()
        x = x.unsqueeze(1)
        return x

    def normalize_batch(self, x):
        x = (x - self.cfg.mean) / self.cfg.std
        return x

    def to_normalized_feature(self, batch_audio):
        x = self.to_log_mel_spec(batch_audio)
        x = self.normalize_batch(x)
        return x

    def encode_lms(self, x, average_per_time_frame=False):
        patch_fbins = self.backbone.grid_size()[0]
        unit_frames = self.cfg.input_size[1]
        patch_frames = self.backbone.patch_size()[1]
        embed_d = self.backbone.patch_embed.proj.out_channels
        n_chunk = (x.shape[-1] + unit_frames - 1) // unit_frames
        pad_frames = (patch_frames - (x.shape[-1] % unit_frames % patch_frames)) % patch_frames
        if pad_frames > 0:
            x = torch.nn.functional.pad(x, (0, pad_frames))

        embeddings = []
        if self.cfg.flat_features:
            # flatten all patch embeddings
            for i in range(n_chunk):
                emb = self.backbone.forward_encoder(x[..., i*unit_frames:(i+1)*unit_frames])
                emb = emb[..., 1:, :]
                if average_per_time_frame:
                    emb = rearrange(emb, 'b (f t) d -> b t d f', f=patch_fbins, d=embed_d).mean(-1)
                embeddings.append(emb)
        else:
            # stack embeddings along time frame
            for i in range(n_chunk):
                emb = self.backbone.forward_encoder(x[..., i*unit_frames:(i+1)*unit_frames])
                emb = emb[..., 1:, :]
                emb = rearrange(emb, 'b (f t) d -> b t (f d)', f=patch_fbins, d=embed_d)
                embeddings.append(emb)
        # concatenate embedding chunks in the time axis
        x = torch.cat(embeddings, axis=-2)
        return x

    def encode(self, batch_audio, average_per_time_frame=False):
        x = self.to_normalized_feature(batch_audio)
        return self.encode_lms(x, average_per_time_frame=average_per_time_frame)

    def forward(self, batch_audio, average_per_time_frame=False):
        x = self.encode(batch_audio, average_per_time_frame=average_per_time_frame)
        if hasattr(self, 'head'):
            x = x.mean(1)  # B, D
            x = self.head_norm(x.unsqueeze(-1)).squeeze(-1)
            x = self.head(x)
        return x

    def get_scene_embeddings(self, batch_audio):
        x = self.encode(batch_audio)
        x = torch.mean(x, dim=1)
        return x

    def get_timestamp_embeddings(self, batch_audio):
        x = self.encode(batch_audio, average_per_time_frame=True)
        ts = get_timestamps(self.cfg, batch_audio, x)
        return x, ts

    def forward_frames(self, batch_audio):
        x, ts = self.get_timestamp_embeddings(batch_audio)
        if hasattr(self, 'head'):
            x = self.head_norm(x.transpose(-1, -2)).transpose(-2, -1)
            x = self.head(x)
        return x, ts

    def encode_clap_audio(self, batch_audio):
        audio_embeddings = self.forward(batch_audio)
        if not hasattr(self.backbone.audio_proj, 'dont_average'):
            audio_embeddings = audio_embeddings.mean(dim=-2)
        audio_embeddings = self.backbone.audio_proj(audio_embeddings)
        return audio_embeddings

    def encode_clap_text(self, batch_text, truncate=False):
        if not hasattr(self, 'text_encoder'):
            self.get_clap_text_encoder()
        text_embeddings = self.text_encoder(batch_text, truncate=truncate)
        if hasattr(self.backbone, 'text_proj'):
            text_embeddings = self.backbone.text_proj(text_embeddings)
        return text_embeddings

    def get_clap_text_encoder(self):
        text_encoder_weight = self.cfg.text_encoder_weight if hasattr(self.cfg, 'text_encoder_weight') else None
        self.text_encoder = get_text_encoder(self.cfg.weight_file, text_encoder_weight=text_encoder_weight)
        self.text_encoder = self.text_encoder.to(next(self.backbone.parameters()).device)


# For the CLAP models

class GTETextEncoder(torch.nn.Module):
    def __init__(self, clip_weight="thenlper/gte-base"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        cache_dir = os.environ.get("HF_HOME")
        self.tokenizer = AutoTokenizer.from_pretrained(clip_weight, cache_dir=cache_dir)
        self.model = AutoModel.from_pretrained(clip_weight, cache_dir=cache_dir)

    def __call__(self, texts, truncate=True, max_length=512):
        def average_pool(last_hidden_states, attention_mask):
            last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
            return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

        device = next(self.model.parameters()).device
        batch_dict = self.tokenizer(texts, max_length=max_length, padding=True, truncation=truncate, return_tensors='pt')
        batch_dict['input_ids'] = batch_dict['input_ids'].to(device)
        batch_dict['token_type_ids'] = batch_dict['token_type_ids'].to(device)
        batch_dict['attention_mask'] = batch_dict['attention_mask'].to(device)
        outputs = self.model.to(device)(**batch_dict)
        embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
        return embeddings


class GTEL15Encoder(torch.nn.Module):
    def __init__(self, clip_weight="Alibaba-NLP/gte-large-en-v1.5"):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        self.model = SentenceTransformer(clip_weight, trust_remote_code=True)

    def __call__(self, texts, **kwargs):
        embeddings = self.model.encode(texts, show_progress_bar=False, convert_to_tensor=True)
        return embeddings


class NVEmbedV2Encoder(torch.nn.Module):
    def __init__(self, clip_weight="nvidia/NV-Embed-v2"):
        # https://huggingface.co/spaces/mteb/leaderboard https://huggingface.co/nvidia/NV-Embed-v2
        # https://arxiv.org/pdf/2405.17428
        from transformers import AutoModel
        super().__init__()
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        self.model = AutoModel.from_pretrained(
            clip_weight, trust_remote_code=True, cache_dir=os.environ.get("HF_HOME")
        )

    def __call__(self, texts, **kwargs):
        embeddings = self.model.encode(texts, instruction="", max_length=32768)
        return embeddings


class CLIPLTextEncoder(torch.nn.Module):
    def __init__(self, clip_weight="ViT-L/14"):
        super().__init__()
        import clip

        device = 'cpu'  # "cuda" if torch.cuda.is_available() else "cpu"
        self.clip = clip
        self.model, _ = clip.load(clip_weight, device=device)

    def __call__(self, texts, truncate=True, max_length=77):
        device = next(self.model.parameters()).device
        tokens = self.clip.tokenize(texts, context_length=max_length, truncate=True).to(device)
        embeddings = self.model.to(device).encode_text(tokens)
        return embeddings


class BertXEncoder(torch.nn.Module):
    def __init__(self, clip_weight="google-bert/bert-base-uncased"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        import os
        os.environ["TOKENIZERS_PARALLELISM"] = "true"  # To suppress warnings.

        cache_dir = os.environ.get("HF_HOME")
        self.tokenizer = AutoTokenizer.from_pretrained(clip_weight, cache_dir=cache_dir)
        self.text_encoder = AutoModel.from_pretrained(clip_weight, cache_dir=cache_dir)

    def forward(self, batch_text, truncate=True, max_length=512):
        device = next(self.text_encoder.parameters()).device
        text_input = self.tokenizer(batch_text,
            padding='longest',
            truncation=truncate,
            max_length=max_length,
            return_tensors="pt").to(device)
        text_feats = self.text_encoder(input_ids=text_input.input_ids,
            attention_mask=text_input.attention_mask)[0]
        text_feats = text_feats[:, 0, :]
        return text_feats


def get_text_encoder(weight, text_encoder_weight=None):
    try:
        clap_type = parse_clap_type(Path(weight).parent.name)
    except:
        clap_type = parse_clap_type(Path(weight).stem)

    if clap_type == 'A':
        text_model = GTETextEncoder()
    if clap_type == 'B':
        text_model = BertXEncoder()
    if clap_type == 'C':
        text_model = CLIPLTextEncoder()
    if clap_type == 'L':
        text_model = GTEL15Encoder()
    if clap_type == 'M':
        text_model = BertXEncoder(clip_weight="google-bert/bert-large-uncased")
    if clap_type == 'N':
        text_model = NVEmbedV2Encoder()
    if clap_type == 'Q':
        text_model = BertXEncoder(clip_weight="FacebookAI/roberta-large")
    if clap_type == 'R':
        text_model = BertXEncoder(clip_weight="FacebookAI/roberta-base")

    if text_encoder_weight is not None:
        weights = torch.load(text_encoder_weight, map_location='cpu', weights_only=False)
        weights = weights['model'] if 'model' in weights else weights
        if any(['module.ar.runtime.' in k for k in weights]):
            print(f' loading EVAR weight {text_encoder_weight} by removing "module.ar.runtime." from keys.')
            renamed = {k.replace('module.ar.runtime.', ''): weights[k] for k in weights}
            weights = renamed
        weights = extract_weight(weights, 'text_encoder.')
        print(f' using model.text_encoder from {text_encoder_weight}')
        text_model.load_state_dict(weights)
    return text_model

## 2. Configuration and path validation

Point `drive_data_path` at the supplied dataset directory, `drive_checkpoint_path` at its approved checkpoint file, and `drive_output_path` at the desired Drive output folder. The loader prints the observed directory contents and validates metadata/audio paths before training.


In [ ]:
import os, sys, math, time, json, shutil, random, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from scipy.stats import spearmanr, pearsonr, kendalltau

CONFIG = {
    "drive_data_path": "/content/drive/MyDrive/XACLE_Challenge_2027_development_dataset",

    "drive_checkpoint_path": (
        "/content/m2d_ckpt/"
        "m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/"
        "checkpoint-30.pth"
    ),

    "drive_output_path": "/content/drive/MyDrive/E34_outputs",
    "copy_data_to_local": True,
    "seed": 7,
    "batch_size": 32,
    "epochs": 6,
    "patience": 2,
    "experiment": "group_dro",  # erm | balanced_erm | group_dro | balanced_group_dro | comparison
    "run_smoke_test": False,
    "run_logo": True,
    "run_final_test": True,  # evaluate once after selection; never used for tuning
    "run_comparison": False,
    "workers": 2,
    "eval_batch_size": 64,
    "head_lr": 1e-3,
    "lora_lr": 5e-4,
    "weight_decay": 1e-2,
    "lora_rank": 8,
    "lora_alpha": 16,
    "audio_lora_blocks": 4,
    "text_lora_layers": 2,
    "coral_weight": 1.0,
    "huber_weight": 0.5,
    "huber_delta": 1.0,
    "pair_weight": 0.5,
    "srcc_weight": 0.5,
    "pair_gap": 0.2,
    "pair_tau": 0.5,
    "srcc_tau": 0.1,
    "mix": 0.5,
    "select_on": "mix",  # coral | reg | mix
    "dro_step_size": 0.01,
    "dro_smoothing": 0.0,
    "real_group_names": ["audiocaps"],
    "run_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
}

CFG = CONFIG  # E33-compatible name used by retained LoRA/runtime helpers.
SR, DUR, N_LEVELS = 16000, 10, 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = META_DIR = WAV_ROOT = CKPT = OUT = None
READY = False

def configure_paths(config=CONFIG):
    global DATA_ROOT, META_DIR, WAV_ROOT, CKPT, OUT, READY
    required = ["drive_data_path", "drive_checkpoint_path", "drive_output_path"]
    missing = [k for k in required if not str(config.get(k, "")).strip()]
    if missing:
        print("Training is disabled. Configure these Drive paths in CONFIG:", ", ".join(missing))
        READY = False
        return False
    data_path = Path(config["drive_data_path"]).expanduser()
    ckpt_path = Path(config["drive_checkpoint_path"]).expanduser()
    out_path = Path(config["drive_output_path"]).expanduser()
    if not data_path.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {data_path}")
    if not ckpt_path.is_file():
        raise FileNotFoundError(f"M2D-CLAP checkpoint file does not exist: {ckpt_path}")
    if not ckpt_path.name.endswith(".pth"):
        raise ValueError("Expected the supplied M2D-CLAP checkpoint as a .pth file.")
    print("Dataset directory entries:", sorted(p.name for p in data_path.iterdir())[:30])
    meta = data_path / "meta_data"
    wavs = data_path / "wav"
    if not meta.is_dir() or not wavs.is_dir():
        raise FileNotFoundError("Expected the supplied dataset directory to contain meta_data/ and wav/; inspect its actual structure and set drive_data_path to that directory.")
    if not (meta / "train_average.csv").is_file():
        observed = sorted(p.name for p in meta.glob("*.csv"))
        raise FileNotFoundError(f"Expected E33 metadata file meta_data/train_average.csv; found CSV files: {observed}")
    if config["copy_data_to_local"]:
        local_root = Path("/content/xacle_data") / data_path.name
        if not (local_root / "meta_data" / "train_average.csv").exists():
            local_root.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(data_path, local_root, dirs_exist_ok=True)
        data_path = local_root
    DATA_ROOT, META_DIR, WAV_ROOT = data_path, data_path / "meta_data", data_path / "wav"
    CKPT, OUT = ckpt_path, out_path
    OUT.mkdir(parents=True, exist_ok=True)
    READY = True
    print(f"Device: {DEVICE}; data: {DATA_ROOT}; checkpoint: {CKPT}; outputs: {OUT}")
    return True

READY = configure_paths()


Dataset directory entries: ['.DS_Store', 'meta_data', 'wav']
Device: cpu; data: /content/xacle_data/XACLE_Challenge_2027_development_dataset; checkpoint: /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth; outputs: /content/drive/MyDrive/E34_outputs


## 3. Dataset inspection and generator groups

The CSV schema and unique `model_name` values are inspected before training. `audiocaps` is treated as a real-audio domain based on the observed E33 metadata; change `real_group_names` only after confirming the supplied dataset semantics. Generator labels are training metadata only and are not passed to the model at inference.


In [ ]:
def load_split(split):
    if not READY:
        raise RuntimeError("Configure all Drive paths before loading data.")
    csv_path = META_DIR / f"{split}_average.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(f"Missing metadata CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    df.columns = [str(c).strip() for c in df.columns]
    wav_col = next((c for c in df.columns if c.lower().startswith(("wav", "wave"))), None)
    score_col = next((c for c in df.columns if "average" in c.lower() and "score" in c.lower()), None)
    required = {"text", "model_name"}
    if wav_col is None or score_col is None or not required.issubset(df.columns):
        raise ValueError(f"Unexpected schema in {csv_path}. Columns found: {df.columns.tolist()}")
    names = df[wav_col].astype(str).str.strip().tolist()
    paths = [WAV_ROOT / split / Path(n.lstrip("/")) for n in names]
    missing_audio = [str(p) for p in paths if not p.is_file()]
    if missing_audio:
        raise FileNotFoundError(f"{len(missing_audio)} audio file(s) missing; examples: {missing_audio[:5]}")
    scores = pd.to_numeric(df[score_col], errors="coerce").to_numpy(np.float32)
    texts = df["text"].fillna("").astype(str).tolist()
    groups = df["model_name"].fillna("__unknown__").astype(str).str.strip().to_numpy()
    if not np.isfinite(scores).all() or not all(texts):
        raise ValueError(f"Invalid score or empty caption found in {csv_path}")
    return {"wavs": paths, "texts": texts, "y": scores, "group": groups,
            "names": names, "generator_id": groups.copy()}

if READY:
    DATA = {s: load_split(s) for s in ("train", "validation")}
    if CONFIG["run_final_test"]:
        DATA["test"] = load_split("test")
    print("Observed metadata columns and group counts:")
    print(pd.Series(DATA["train"]["group"]).value_counts().to_string())
    REAL_GROUPS = set(map(str, CONFIG["real_group_names"]))
    SYNTHETIC_GENERATORS = sorted(set(DATA["train"]["group"]) - REAL_GROUPS)
    if not SYNTHETIC_GENERATORS:
        raise ValueError("No eligible synthetic generator groups found after excluding configured real-audio groups.")
    GROUP_NAMES = sorted(set(DATA["train"]["group"]))
    GROUP_TO_ID = {name: i for i, name in enumerate(GROUP_NAMES)}
    for split_data in DATA.values():
        unknown = set(split_data["group"]) - set(GROUP_TO_ID)
        if unknown:
            raise ValueError(f"Groups absent from training split: {sorted(unknown)}")
        split_data["group_idx"] = np.asarray([GROUP_TO_ID[g] for g in split_data["group"]], dtype=np.int64)
else:
    DATA = {}
    REAL_GROUPS, SYNTHETIC_GENERATORS, GROUP_NAMES, GROUP_TO_ID = set(), [], [], {}


Observed metadata columns and group counts:
audiocaps         2500
stable_audio       750
audiogen           750
make-an-audio2     750
tangoflux          750
audioldm2          500
tango              500
tango2             500
audioldm           500


## 4. LoRA injection (retained from E33)

The M2D audio branch uses the last four ViT blocks and the BERT text branch uses its last two layers, with rank 8 and alpha 16. Backbone weights stay frozen.


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: int):
        super().__init__()
        self.base = base
        for p in base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

def _get(module, name):
    return module[int(name)] if name.isdigit() and not hasattr(module, name) else getattr(module, name)

def _set(module, name, value):
    if name.isdigit() and not hasattr(module, name): module[int(name)] = value
    else: setattr(module, name, value)

def wrap_linear(root, path, r, alpha):
    *parents, last = path.split(".")
    module = root
    for part in parents: module = _get(module, part)
    child = _get(module, last)
    if not isinstance(child, nn.Linear): raise TypeError(f"Expected Linear at {path}")
    _set(module, last, LoRALinear(child, r, alpha))
    return path

def inject_lora(model):
    done = []
    blocks = model.backbone.blocks
    for i in range(max(0, len(blocks)-CONFIG["audio_lora_blocks"]), len(blocks)):
        for sub in ("attn.qkv", "attn.proj"):
            done.append("audio." + wrap_linear(model.backbone, f"blocks.{i}.{sub}", CONFIG["lora_rank"], CONFIG["lora_alpha"]))
    bert = model.text_encoder.text_encoder
    layers = len(bert.encoder.layer)
    for i in range(max(0, layers-CONFIG["text_lora_layers"]), layers):
        for sub in ("attention.self.query", "attention.self.value"):
            done.append("text." + wrap_linear(bert, f"encoder.layer.{i}.{sub}", CONFIG["lora_rank"], CONFIG["lora_alpha"]))
    return done


## 5. E33 head, fold-local normalization, and loss functions

The head and interaction features are unchanged. `label_stats` are computed only from the training indices of the current fit, including each LOGO inner fit.


In [ ]:
def label_stats(data, idx):
    y = np.asarray(data["y"])[np.asarray(idx, dtype=int)]
    sd = float(np.std(y, ddof=1)) if len(y) > 1 else 1.0
    return float(np.mean(y)), max(sd, 1e-8)

class Head(nn.Module):
    def __init__(self, in_dim=4*768+1, hidden=1024, mid=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden, mid), nn.ReLU(), nn.Linear(mid, N_LEVELS+1))
        with torch.no_grad():
            self.net[-1].weight[N_LEVELS].mul_(0.1)
            self.net[-1].bias[N_LEVELS].zero_()
    def forward(self, x): return self.net(x)

def inter_feats(a, t):
    a = (a-a.mean(-1, keepdim=True))/(a.std(-1, keepdim=True)+1e-6)
    t = (t-t.mean(-1, keepdim=True))/(t.std(-1, keepdim=True)+1e-6)
    return torch.cat([a, t, a*t, (a-t).abs(), (a*t).sum(-1, keepdim=True)], -1).float()

def score_parts(out, stats):
    mu, sd = stats
    out = out.float()
    zc = (torch.sigmoid(out[:, :N_LEVELS]).sum(-1)-mu)/sd
    zr = out[:, N_LEVELS]
    return {"coral": zc, "reg": zr, "mix": (1-CONFIG["mix"])*zc+CONFIG["mix"]*zr}

def to_score(out, stats, key=None):
    mu, sd = stats
    return mu + sd*score_parts(out, stats)[key or CONFIG["select_on"]]

def coral_per_sample(out, y):
    logits = out[:, :N_LEVELS].float()
    level = torch.clamp(torch.round(y.float()), 0, 10)
    target = (level.unsqueeze(1) > torch.arange(N_LEVELS, device=y.device).float()).float()
    return F.binary_cross_entropy_with_logits(logits, target, reduction="none").mean(1)

def huber_per_sample(out, y, stats):
    z = (y.float()-stats[0])/stats[1]
    return F.huber_loss(out[:, N_LEVELS].float(), z, delta=CONFIG["huber_delta"], reduction="none")

def soft_rank(x, tau):
    return torch.sigmoid((x.unsqueeze(1)-x.unsqueeze(0))/tau).sum(1)

def pearson_torch(a, b, eps=1e-8):
    a, b = a-a.mean(), b-b.mean()
    return (a*b).sum()/(a.norm()*b.norm()+eps)

def srcc_loss(s, z, tau):
    with torch.no_grad(): rz = soft_rank(z, 1e-4)
    return 1.0-pearson_torch(soft_rank(s, tau), rz)

def pair_loss(s, z, gap, tau):
    dz = z.unsqueeze(1)-z.unsqueeze(0)
    ds = s.unsqueeze(1)-s.unsqueeze(0)
    mask = dz > gap
    if not bool(mask.any()): return s.sum()*0.0
    w = dz[mask].clamp(max=2.0)
    w = w/w.mean()
    return (w*F.softplus(-ds[mask]/tau)).mean()

def total_loss(out, y, stats):
    sp = score_parts(out, stats)
    coral = coral_per_sample(out, y).mean()
    huber = huber_per_sample(out, y, stats).mean()
    z = (y.float()-stats[0])/stats[1]
    pair = pair_loss(sp["mix"], z, CONFIG["pair_gap"], CONFIG["pair_tau"])
    srcc = srcc_loss(sp["mix"], z, CONFIG["srcc_tau"])
    total = CONFIG["coral_weight"]*coral + CONFIG["huber_weight"]*huber + CONFIG["pair_weight"]*pair + CONFIG["srcc_weight"]*srcc
    return total, {"coral": coral, "huber": huber, "pair": pair, "srcc": srcc}


## 6. Audio dataset, inference, and metrics

Audio is read as mono float32, resampled to 16 kHz when needed, and padded/truncated to the original E33 ten-second window. Group IDs are returned for training bookkeeping and are not given to the model encoder.


In [ ]:
def read_wav(path):
    wave, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wave = wave.mean(1)
    if sr != SR:
        from scipy.signal import resample_poly
        from math import gcd
        g = gcd(int(sr), SR)
        wave = resample_poly(wave, SR//g, int(sr)//g).astype(np.float32)
    n = SR*DUR
    return np.pad(wave, (0, max(0, n-len(wave))))[:n]

class PairDS(Dataset):
    def __init__(self, data, idx): self.data, self.idx = data, np.asarray(idx, dtype=int)
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = int(self.idx[k])
        return (torch.from_numpy(read_wav(self.data["wavs"][i])), self.data["texts"][i],
                float(self.data["y"][i]), int(self.data["group_idx"][i]), i)

def collate(batch):
    return (torch.stack([x[0] for x in batch]), [x[1] for x in batch],
            torch.tensor([x[2] for x in batch], dtype=torch.float32),
            torch.tensor([x[3] for x in batch], dtype=torch.long),
            torch.tensor([x[4] for x in batch], dtype=torch.long))

def encode_pair(model, wavs, texts):
    x = model.to_normalized_feature(wavs)
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        frames = model.encode_lms(x)
    a = model.backbone.audio_proj(frames.float())
    t = model.encode_clap_text(texts, truncate=True)
    return F.normalize(a.float(), dim=-1), F.normalize(t.float(), dim=-1)

def safe_corr(fn, y, p):
    if len(y) < 2 or np.ptp(y) == 0 or np.ptp(p) == 0: return float("nan")
    try: return float(fn(y, p).statistic)
    except Exception: return float("nan")

def metrics(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    return {"srcc": safe_corr(spearmanr, y, p), "lcc": safe_corr(pearsonr, y, p),
            "ktau": safe_corr(kendalltau, y, p), "mse": float(np.mean((y-p)**2)) if len(y) else float("nan"),
            "n": int(len(y))}

@torch.no_grad()
def predict_parts(model, head, data, idx, stats):
    model.eval(); head.eval()
    dl = DataLoader(PairDS(data, idx), batch_size=CONFIG["eval_batch_size"], shuffle=False,
                    num_workers=CONFIG["workers"], collate_fn=collate, pin_memory=DEVICE.type == "cuda")
    outs, indices = [], []
    for w, t, _, _, original_idx in dl:
        a, tt = encode_pair(model, w.to(DEVICE, non_blocking=True), t)
        outs.append(head(inter_feats(a, tt)).float().cpu())
        indices.extend(original_idx.numpy().tolist())
    out = torch.cat(outs) if outs else torch.empty((0, N_LEVELS+1))
    return {k: (stats[0]+stats[1]*v).numpy() for k, v in score_parts(out, stats).items()}, np.asarray(indices)


## 7. Group DRO and balanced sampling

For each minibatch, group losses are means of `CORAL_i + 0.5 × Huber_i`. Persistent log-weights update with `log q_g ← log q_g + η L_g` and are normalized with log-sum-exp. Only groups present in a batch update; absent groups retain their weights and are not assigned zero loss. For the minibatch objective, weights are renormalized over present groups. Pairwise and soft-Spearman losses are computed once for the full batch, since they depend on sample relationships.

Balanced sampling uses inverse-frequency sample weights with replacement. It changes the batch sampling distribution and is therefore reported separately from Group DRO.


In [ ]:
class GroupDROState:
    def __init__(self, group_ids, step_size=0.01, smoothing=0.0):
        self.group_ids = tuple(sorted(set(map(int, group_ids))))
        if not self.group_ids: raise ValueError("GroupDROState requires at least one group")
        self.step_size, self.smoothing = float(step_size), float(smoothing)
        self.log_q = torch.full((max(self.group_ids)+1,), -math.log(len(self.group_ids)), dtype=torch.float64)
        absent = sorted(set(range(len(self.log_q.tolist()))) - set(self.group_ids))
        if absent: self.log_q[absent] = -torch.inf
        self.history = []
    @property
    def q(self):
        vals = torch.softmax(self.log_q, dim=0)
        return vals
    def update(self, group_losses):
        if not group_losses: raise ValueError("Cannot update Group DRO with an empty group-loss mapping")
        with torch.no_grad():
            for gid, loss in group_losses.items():
                value = float(torch.as_tensor(loss).detach().cpu())
                old = float(self.log_q[int(gid)])
                self.log_q[int(gid)] = old + self.step_size*value
            self.log_q -= torch.logsumexp(self.log_q, dim=0)
            if self.smoothing > 0:
                q = torch.softmax(self.log_q, dim=0)
                uniform = torch.zeros_like(q)
                active = torch.tensor(self.group_ids, dtype=torch.long)
                uniform[active] = 1.0/len(self.group_ids)
                q = (1-self.smoothing)*q + self.smoothing*uniform
                self.log_q = torch.where(q > 0, q.log(), torch.full_like(q, -torch.inf))
        self.history.append(self.q.cpu().tolist())
    def objective(self, group_losses):
        gids = torch.tensor(list(group_losses), dtype=torch.long)
        weights = self.q[gids]
        weights = weights/weights.sum().clamp_min(1e-12)
        losses = torch.stack([group_losses[int(g)] for g in gids.tolist()])
        return (weights.to(losses.device, dtype=losses.dtype)*losses).sum()

def group_loss_map(out, y, group_ids, stats):
    per_sample = coral_per_sample(out, y) + CONFIG["huber_weight"]*huber_per_sample(out, y, stats)
    return {int(g): per_sample[group_ids == int(g)].mean() for g in torch.unique(group_ids).tolist()}

def balanced_sampler(data, train_idx):
    groups = np.asarray(data["group_idx"])[np.asarray(train_idx, dtype=int)]
    counts = pd.Series(groups).value_counts().to_dict()
    weights = torch.tensor([1.0/counts[int(g)] for g in groups], dtype=torch.double)
    return WeightedRandomSampler(weights, num_samples=len(train_idx), replacement=True)

def split_is_disjoint(train_idx, held_idx):
    return not bool(set(map(int, train_idx)) & set(map(int, held_idx)))


## 8. Model creation and reusable training functions

Every call to `train_one` builds a fresh M2D-CLAP model, LoRA adapters, head, optimizer, scaler, scheduler, and Group DRO state. Validation SRCC controls early stopping for the main run and LOGO inner validation only.


In [ ]:
def build_model():
    if not READY: raise RuntimeError("Configure Drive paths before building a model.")
    os.environ.setdefault("HF_HOME", "/tmp/hf_cache")
    model = PortableM2D(str(CKPT), flat_features=True)
    model.get_clap_text_encoder()
    model.requires_grad_(False)
    lora_names = inject_lora(model)
    return model.to(DEVICE), lora_names

def lr_factor(step, total, warm=0.1):
    warm_steps = max(1, int(round(warm*total)))
    if step < warm_steps: return (step+1)/warm_steps
    return 0.5*(1+math.cos(math.pi*min(1.0, (step-warm_steps)/max(1,total-warm_steps))))

def predict_selected(model, head, data, idx, stats):
    parts, _ = predict_parts(model, head, data, idx, stats)
    return parts[CONFIG["select_on"]]

def train_one(data, train_idx, mode, seed, val_data=None, val_idx=None, fixed_epochs=None,
              stats=None, output_dir=None, tag=""):
    if mode not in {"erm", "balanced_erm", "group_dro", "balanced_group_dro"}:
        raise ValueError(f"Unsupported training mode: {mode}")
    train_idx = np.asarray(train_idx, dtype=int)
    if stats is None: stats = label_stats(data, train_idx)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model, lora_names = build_model()
    head = Head().to(DEVICE)
    lora_params = [p for p in model.parameters() if p.requires_grad]
    frozen_trainable = [n for n,p in model.named_parameters() if p.requires_grad and ".base." in n]
    if frozen_trainable: raise AssertionError(f"Frozen backbone unexpectedly trainable: {frozen_trainable[:5]}")
    opt = torch.optim.AdamW([
        {"params": head.parameters(), "lr": CONFIG["head_lr"], "weight_decay": CONFIG["weight_decay"]},
        {"params": lora_params, "lr": CONFIG["lora_lr"], "weight_decay": 0.0}],
        betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    sampler = balanced_sampler(data, train_idx) if mode in {"balanced_erm", "balanced_group_dro"} else None
    dl = DataLoader(PairDS(data, train_idx), batch_size=CONFIG["batch_size"], shuffle=sampler is None,
                    sampler=sampler, drop_last=False,
                    num_workers=CONFIG["workers"], collate_fn=collate,
                    persistent_workers=CONFIG["workers"] > 0, pin_memory=DEVICE.type == "cuda")
    epochs = int(fixed_epochs or CONFIG["epochs"])
    scheduler_step, total_steps = 0, max(1, len(dl)*epochs)
    base_lr = [CONFIG["head_lr"], CONFIG["lora_lr"]]
    dro = None
    if mode in {"group_dro", "balanced_group_dro"}:
        train_group_ids = sorted(set(np.asarray(data["group_idx"])[train_idx].tolist()))
        dro = GroupDROState(train_group_ids, CONFIG["dro_step_size"], CONFIG["dro_smoothing"])
    best, best_ep, bad = -float("inf"), epochs, 0
    best_state, history = None, []
    for ep in range(1, epochs+1):
        model.train(); head.train(); start = time.time()
        totals, nb, group_sum, group_n = {}, 0, {}, {}
        for w, texts, y, gid, _ in dl:
            w, y, gid = w.to(DEVICE, non_blocking=True), y.to(DEVICE), gid.to(DEVICE)
            a, t = encode_pair(model, w, texts)
            out = head(inter_feats(a, t))
            if dro is None:
                loss, parts = total_loss(out, y, stats)
            else:
                g_losses = group_loss_map(out, y, gid, stats)
                dro.update({g: v.detach() for g,v in g_losses.items()})
                pair = pair_loss(score_parts(out, stats)["mix"], (y-stats[0])/stats[1], CONFIG["pair_gap"], CONFIG["pair_tau"])
                srcc = srcc_loss(score_parts(out, stats)["mix"], (y-stats[0])/stats[1], CONFIG["srcc_tau"])
                loss = dro.objective(g_losses) + CONFIG["pair_weight"]*pair + CONFIG["srcc_weight"]*srcc
                parts = {"group_objective": dro.objective(g_losses), "pair": pair, "srcc": srcc}
                for g,v in g_losses.items():
                    group_sum[g] = group_sum.get(g, 0.0) + float(v.detach())
                    group_n[g] = group_n.get(g, 0) + 1
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss in {mode} epoch {ep}")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([*head.parameters(), *lora_params], 1.0)
            if not any(p.grad is not None and torch.isfinite(p.grad).all() for p in lora_params):
                raise RuntimeError("LoRA parameters received no finite gradients.")
            scaler.step(opt); scaler.update()
            scheduler_step += 1
            factor = lr_factor(scheduler_step, total_steps)
            for group, lr in zip(opt.param_groups, base_lr): group["lr"] = lr*factor
            totals["objective"] = totals.get("objective", 0.0)+float(loss.detach())
            for k,v in parts.items(): totals[k] = totals.get(k, 0.0)+float(v.detach())
            nb += 1
        row = {"epoch": ep, "train_loss": totals.get("objective", 0.0)/max(1,nb),
               **{f"train_{k}":v/max(1,nb) for k,v in totals.items()}, "seconds": time.time()-start}
        if dro is not None:
            row["group_losses"] = {GROUP_NAMES[g]: group_sum[g]/group_n[g] for g in group_sum}
            row["group_weights"] = {GROUP_NAMES[g]: float(dro.q[g]) for g in dro.group_ids}
        if val_data is not None and val_idx is not None:
            pred = predict_selected(model, head, val_data, val_idx, stats)
            val_y = np.asarray(val_data["y"])[np.asarray(val_idx, dtype=int)]
            val_srcc = metrics(val_y, pred)["srcc"]
            row["val_srcc"] = val_srcc
            history.append(row)
            print(f"{tag}{mode} ep {ep}/{epochs}: loss={row['train_loss']:.4f}, val SRCC={val_srcc:.4f}, {row['seconds']:.0f}s")
            if np.isfinite(val_srcc) and val_srcc > best+1e-5:
                best, best_ep, bad = val_srcc, ep, 0
                best_state = ({k:v.detach().clone() for k,v in head.state_dict().items()},
                              {n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad})
            else:
                bad += 1
            if fixed_epochs is None and bad >= CONFIG["patience"]:
                print(f"{tag}{mode}: early stopping at epoch {ep}")
                break
        else:
            history.append(row)
            print(f"{tag}{mode} ep {ep}/{epochs}: loss={row['train_loss']:.4f}, {row['seconds']:.0f}s")
    if best_state is not None:
        head.load_state_dict(best_state[0])
        with torch.no_grad():
            for n,p in model.named_parameters():
                if p.requires_grad and n in best_state[1]: p.copy_(best_state[1][n])
    if output_dir is not None:
        output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
        save_checkpoint(output_dir/"best.pt", model, head, stats, seed, mode, best_ep, dro)
        (output_dir/"history.json").write_text(json.dumps(history, indent=2, default=float), encoding="utf-8")
    return {"model":model, "head":head, "lora_names":lora_names, "best_epoch":best_ep,
            "history":history, "stats":stats, "dro":dro}


## 9. Checkpoints and standalone inference

Adapter parameters are saved by parameter name, along with head weights, configuration, seed, normalization statistics, group map, selected epoch, and Group DRO state. Inference takes only waveform and caption; `generator_id` is never required.


In [ ]:
def save_checkpoint(path, model, head, stats, seed, mode, best_epoch, dro=None, metrics_record=None):
    adapter = {n:p.detach().cpu() for n,p in model.named_parameters() if p.requires_grad}
    payload = {"adapter": adapter, "head": {k:v.detach().cpu() for k,v in head.state_dict().items()},
               "config": {k:v for k,v in CONFIG.items() if k != "run_id"}, "seed": int(seed),
               "mode": mode, "label_stats": list(map(float,stats)), "group_names": list(GROUP_NAMES),
               "group_to_id": dict(GROUP_TO_ID), "best_epoch": int(best_epoch),
               "dro_log_q": None if dro is None else dro.log_q.cpu().tolist(),
               "metrics": metrics_record or {}, "checkpoint_path": str(CKPT)}
    torch.save(payload, path)

def load_e34_checkpoint(path, model=None):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if model is None: model, _ = build_model()
    named = dict(model.named_parameters())
    with torch.no_grad():
        for name, value in ck["adapter"].items():
            if name not in named: raise KeyError(f"Adapter parameter missing in reconstructed model: {name}")
            named[name].copy_(value.to(named[name].device, dtype=named[name].dtype))
    head = Head().to(DEVICE); head.load_state_dict(ck["head"])
    return model, head, tuple(ck["label_stats"]), ck

def predict_one(model, head, waveform, caption, stats):
    """Predict one alignment score from waveform and caption; no group ID is accepted."""
    wave = np.asarray(waveform, dtype=np.float32)
    if wave.ndim == 2: wave = wave.mean(axis=1)
    wave = np.pad(wave, (0,max(0,SR*DUR-len(wave))))[:SR*DUR]
    with torch.no_grad():
        a,t = encode_pair(model, torch.from_numpy(wave[None]).to(DEVICE), [str(caption)])
        out = head(inter_feats(a,t))
        return float(to_score(out, stats)[0].cpu())


## 10. Synthetic implementation checks

These tests use synthetic tensors only to validate implementation behavior; they do not produce challenge metrics. The optional toggle runs these checks independently of Drive data availability.


In [ ]:
def run_synthetic_unit_tests():
    # Group weights stay normalized, nonnegative, and increase for the higher-loss group.
    state = GroupDROState([0,1,2], step_size=1.0)
    state.update({0: torch.tensor(0.1), 1: torch.tensor(2.0)})  # group 2 is absent and untouched
    q = state.q
    assert torch.all(q >= 0) and torch.isclose(q.sum(), torch.tensor(1.0, dtype=q.dtype))
    assert q[1] > q[0] and q[2] < q[1]
    before_absent_odds = float(q[2]/q[1])
    state.update({0: torch.tensor(0.1)})
    assert abs(float(state.q[2]/state.q[1])-before_absent_odds) < 1e-12  # absent group receives no zero-loss update
    # Balanced sampler covers only requested rows and gives inverse-frequency weights.
    toy = {"group_idx": np.array([0,0,0,1,1,2])}
    sampler = balanced_sampler(toy, np.arange(6))
    sampled = list(iter(sampler))
    assert len(sampled) == 6 and all(0 <= i < 6 for i in sampled)
    # Per-sample objectives are finite; each group mean includes only its own examples.
    out = torch.zeros(4, N_LEVELS+1, requires_grad=True)
    y = torch.tensor([1.,2.,8.,9.]); stats = (5.0, 2.0)
    gids = torch.tensor([0,0,1,1])
    gm = group_loss_map(out, y, gids, stats)
    assert set(gm) == {0,1} and all(torch.isfinite(v) for v in gm.values())
    obj = state.objective({0:gm[0],1:gm[1]}); obj.backward()
    assert out.grad is not None and torch.isfinite(out.grad).all()
    # Split leakage audit and safe metrics for constant labels.
    assert split_is_disjoint([0,1,2],[3,4]) and not split_is_disjoint([0,2],[2,3])
    assert np.isnan(metrics([1,1,1],[0,1,2])["srcc"])
    print("Synthetic checks passed: DRO weighting, absent groups, balanced sampling, loss gradients, split disjointness, and constant-score metrics.")

if CONFIG["run_smoke_test"]:
    run_synthetic_unit_tests()


## 11. Main controlled comparison

The primary `group_dro` setting runs E0 (ERM) and E2 (Group DRO) with identical data, seed, backbone, head, batch size, and epoch/early-stopping policy. `comparison` adds E1 balanced ERM and E3 balanced Group DRO. Main validation uses only the supplied validation split. Test labels are not read by this cell.


In [ ]:
def run_main_mode(mode):
    if not READY: raise RuntimeError("No training: configure all required Drive paths first.")
    train_idx = np.arange(len(DATA["train"]["y"]))
    stats = label_stats(DATA["train"], train_idx)
    run_dir = OUT/"checkpoints"/mode/CONFIG["run_id"]
    fit = train_one(DATA["train"], train_idx, mode, CONFIG["seed"],
                    val_data=DATA["validation"], val_idx=np.arange(len(DATA["validation"]["y"])),
                    stats=stats, output_dir=run_dir, tag="main/")
    preds, order = predict_parts(fit["model"], fit["head"], DATA["validation"],
                                 np.arange(len(DATA["validation"]["y"])), stats)
    row = {"mode":mode, "seed":CONFIG["seed"], "best_epoch":fit["best_epoch"],
           "val_metrics":metrics(DATA["validation"]["y"], preds[CONFIG["select_on"]]),
           "config":{k:v for k,v in CONFIG.items() if k not in {"run_id"}},
           "label_stats":list(stats)}
    outdir = OUT/"predictions"/mode/CONFIG["run_id"]
    outdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"wav_file_name":np.asarray(DATA["validation"]["names"])[order],
                  "prediction":preds[CONFIG["select_on"]],
                  **{f"pred_{k}":v for k,v in preds.items()}}).to_csv(outdir/"validation.csv", index=False)
    (OUT/"logs").mkdir(parents=True, exist_ok=True)
    (OUT/"logs"/f"{mode}_{CONFIG['run_id']}.json").write_text(json.dumps(row,indent=2,default=float),encoding="utf-8")
    return fit, row

def run_controlled_experiment():
    if not READY:
        print("Skipped training because one or more Drive paths are empty.")
        return {}
    modes = {"erm":["erm"], "balanced_erm":["balanced_erm"],
             "group_dro":["erm","group_dro"],
             "balanced_group_dro":["balanced_group_dro"],
             "comparison":["erm","balanced_erm","group_dro","balanced_group_dro"]}
    if CONFIG["experiment"] not in modes: raise ValueError(f"Unknown CONFIG['experiment']: {CONFIG['experiment']}")
    results = {}
    for mode in modes[CONFIG["experiment"]]:
        fit, row = run_main_mode(mode)
        results[mode] = {"fit":fit, "row":row}
    comparison_rows = [{"mode":m, "seed":v["row"]["seed"], "best_epoch":v["row"]["best_epoch"],
                        **v["row"]["val_metrics"]} for m,v in results.items()]
    metrics_dir = OUT/"metrics"; metrics_dir.mkdir(parents=True, exist_ok=True)
    comp = pd.DataFrame(comparison_rows)
    comp.to_csv(metrics_dir/f"comparison_{CONFIG['run_id']}.csv", index=False)
    return results

if READY:
    MAIN_RESULTS = run_controlled_experiment()
else:
    MAIN_RESULTS = {}


 using 166 parameters from /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth
 (included audio_proj params: ['audio_proj.sem_token', 'audio_proj.sem_blocks.0.norm1.weight', 'audio_proj.sem_blocks.0.norm1.bias', 'audio_proj.sem_blocks.0.attn.qkv.weight', 'audio_proj.sem_blocks.0.attn.qkv.bias']
 (included text_proj params: []
 (dropped: [] )
<All keys matched successfully>


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 using model.text_encoder from /content/m2d_ckpt/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth


## 12. Leave-One-Generator-Out evaluation

For each synthetic generator, the outer held-out group is excluded from all fitting. An inner generator from the remaining training groups is reserved for early stopping; then a newly initialized model and optimizer train on all outer-training groups for the selected epoch count. Label statistics are recomputed from each fit's training indices. Separate, timestamped fold directories prevent overwrites. The real-audio group is never reported as a synthetic-generator fold.


In [ ]:
def run_logo(mode="group_dro"):
    if not READY: raise RuntimeError("Configure Drive paths before LOGO.")
    train = DATA["train"]
    if len(SYNTHETIC_GENERATORS) < 2: raise ValueError("LOGO requires at least two eligible generator groups for inner validation.")
    fold_rows = []
    run_root = OUT/"checkpoints"/mode/CONFIG["run_id"]/"logo"
    for outer_gen in SYNTHETIC_GENERATORS:
        held_idx = np.where(train["group"] == outer_gen)[0]
        keep_idx = np.where(train["group"] != outer_gen)[0]
        if not split_is_disjoint(keep_idx, held_idx): raise AssertionError(f"Outer leakage in LOGO fold {outer_gen}")
        inner_candidates = sorted(set(train["group"][keep_idx]) - REAL_GROUPS)
        if not inner_candidates: raise ValueError(f"No inner validation generator available for fold {outer_gen}")
        inner_gen = inner_candidates[0]
        inner_val_idx = np.where(train["group"] == inner_gen)[0]
        inner_train_idx = np.where((train["group"] != inner_gen) & (train["group"] != outer_gen))[0]
        if not split_is_disjoint(inner_train_idx, inner_val_idx) or not split_is_disjoint(inner_train_idx, held_idx):
            raise AssertionError(f"Inner/outer leakage in LOGO fold {outer_gen}")
        inner_stats = label_stats(train, inner_train_idx)
        inner = train_one(train, inner_train_idx, mode, CONFIG["seed"], val_data=train,
                          val_idx=inner_val_idx, stats=inner_stats, tag=f"LOGO[{outer_gen}]/select/")
        selected_epochs = inner["best_epoch"]
        del inner
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        outer_stats = label_stats(train, keep_idx)
        fold_dir = run_root/str(outer_gen)
        fit = train_one(train, keep_idx, mode, CONFIG["seed"], fixed_epochs=selected_epochs,
                        stats=outer_stats, output_dir=fold_dir, tag=f"LOGO[{outer_gen}]/fit/")
        predictions, order = predict_parts(fit["model"], fit["head"], train, held_idx, outer_stats)
        selected = predictions[CONFIG["select_on"]]
        fold_metrics = metrics(train["y"][held_idx], selected)
        pred_dir = OUT/"predictions"/mode/CONFIG["run_id"]/"logo"/str(outer_gen)
        pred_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({"wav_file_name":np.asarray(train["names"])[order], "prediction":selected,
                      "ground_truth":train["y"][order]}).to_csv(pred_dir/"predictions.csv", index=False)
        fold_rows.append({"generator":outer_gen, "inner_validation_generator":inner_gen,
                          "selected_epoch":selected_epochs, **fold_metrics})
        del fit
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    df = pd.DataFrame(fold_rows)
    valid = df.loc[np.isfinite(df["srcc"]), "srcc"]
    summary = {"mode":mode, "run_id":CONFIG["run_id"], "valid_groups":int(len(valid)),
               "logo_macro_srcc":float(valid.mean()) if len(valid) else float("nan"),
               "worst_group_srcc":float(valid.min()) if len(valid) else float("nan"),
               "group_srcc_std":float(valid.std(ddof=0)) if len(valid) else float("nan")}
    metrics_dir = OUT/"metrics"; metrics_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(metrics_dir/f"logo_{mode}_{CONFIG['run_id']}.csv", index=False)
    (metrics_dir/f"logo_summary_{mode}_{CONFIG['run_id']}.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
    print(df.to_string(index=False)); print(json.dumps(summary,indent=2))
    return df, summary

if READY and CONFIG["run_logo"]:
    LOGO_RESULTS = {mode:run_logo(mode) for mode in MAIN_RESULTS}
else:
    LOGO_RESULTS = {}


## 13. Optional final test evaluation

This cell is separate and disabled by default. Run it only after the method and checkpoint have been selected using validation/LOGO. It reports final test metrics once and does not feed them back into training or selection.


In [ ]:
if READY and CONFIG["run_final_test"]:
    test_rows = []
    for mode, result in MAIN_RESULTS.items():
        fit = result["fit"]
        pred, order = predict_parts(fit["model"], fit["head"], DATA["test"], np.arange(len(DATA["test"]["y"])), fit["stats"])
        selected = pred[CONFIG["select_on"]]
        test_rows.append({"mode":mode, **metrics(DATA["test"]["y"], selected)})
        pred_dir = OUT/"predictions"/mode/CONFIG["run_id"]; pred_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({"wav_file_name":np.asarray(DATA["test"]["names"])[order], "prediction":selected,
                      **{f"pred_{k}":v for k,v in pred.items()}}).to_csv(pred_dir/"test_FINAL.csv", index=False)
    pd.DataFrame(test_rows).to_csv(OUT/"metrics"/f"test_FINAL_{CONFIG['run_id']}.csv",index=False)


In [ ]:
display(pd.read_csv(OUT / "metrics" / f"test_FINAL_{CONFIG['run_id']}.csv"))

## 14. Visualizations from actual experiment artifacts

Plots are generated only from saved training histories, LOGO results, and predictions. Empty or unrun experiments produce no synthetic curves.


In [ ]:
import matplotlib.pyplot as plt

def plot_experiment_artifacts(main_results=None, logo_results=None):
    if not READY: print("Configure paths before plotting saved artifacts."); return
    main_results = main_results or {}
    logo_results = logo_results or {}
    fig_dir = OUT/"figures"; fig_dir.mkdir(parents=True, exist_ok=True)
    if main_results:
        fig, axes = plt.subplots(1,2,figsize=(12,4))
        for mode,res in main_results.items():
            hist = pd.DataFrame(res["fit"]["history"])
            if "train_loss" in hist: axes[0].plot(hist["epoch"],hist["train_loss"],marker="o",label=mode)
            if "val_srcc" in hist: axes[1].plot(hist["epoch"],hist["val_srcc"],marker="o",label=mode)
        axes[0].set(title="Training objective",xlabel="Epoch",ylabel="Loss")
        axes[1].set(title="Validation SRCC",xlabel="Epoch",ylabel="SRCC")
        for ax in axes: ax.legend(); ax.grid(alpha=.25)
        fig.tight_layout(); fig.savefig(fig_dir/f"training_{CONFIG['run_id']}.png",dpi=160); plt.close(fig)
        # Generator-specific losses and DRO weights from genuine training history.
        for mode,res in main_results.items():
            hist = res["fit"]["history"]
            if any("group_losses" in r for r in hist):
                fig,axes=plt.subplots(1,2,figsize=(13,4))
                for metric,ax,title in (("group_losses",axes[0],"Generator/domain losses"),("group_weights",axes[1],"Group DRO weights")):
                    keys=sorted(set().union(*(r.get(metric,{}).keys() for r in hist)))
                    for key in keys: ax.plot([r["epoch"] for r in hist if key in r.get(metric,{})],[r[metric][key] for r in hist if key in r.get(metric,{})],marker="o",label=key)
                    ax.set(title=title,xlabel="Epoch"); ax.legend(fontsize=7); ax.grid(alpha=.25)
                fig.tight_layout(); fig.savefig(fig_dir/f"group_dynamics_{mode}_{CONFIG['run_id']}.png",dpi=160); plt.close(fig)
        if len(main_results)>1:
            fig,ax=plt.subplots(figsize=(7,4))
            names=list(main_results); vals=[main_results[m]["row"]["val_metrics"]["srcc"] for m in names]
            ax.bar(names,vals); ax.set(ylabel="Validation SRCC",title="Controlled method comparison"); ax.grid(axis="y",alpha=.25)
            fig.tight_layout(); fig.savefig(fig_dir/f"comparison_{CONFIG['run_id']}.png",dpi=160); plt.close(fig)
    for mode,(fold_df,summary) in logo_results.items():
        if len(fold_df):
            fig,ax=plt.subplots(figsize=(10,4)); ax.bar(fold_df["generator"],fold_df["srcc"])
            ax.axhline(summary["logo_macro_srcc"],color="black",linestyle="--",label="Macro SRCC")
            ax.set(title=f"LOGO SRCC by generator — {mode}",ylabel="SRCC"); ax.tick_params(axis="x",rotation=35); ax.legend()
            fig.tight_layout(); fig.savefig(fig_dir/f"logo_{mode}_{CONFIG['run_id']}.png",dpi=160); plt.close(fig)

plot_experiment_artifacts(MAIN_RESULTS, LOGO_RESULTS)


## 15. Output layout and interpretation

Artifacts are written below the configured Drive output directory:

```text
checkpoints/{method}/{run_id}/
metrics/comparison_{run_id}.csv
metrics/logo_{method}_{run_id}.csv
predictions/{method}/{run_id}/
figures/
logs/{method}_{run_id}.json
```

For a stability study, rerun the controlled comparison with seeds 21 and 42 and distinct `run_id` values. Aggregate only metrics across independent runs; do not average model predictions. Report the number of valid LOGO groups alongside macro SRCC, worst-group SRCC, and group standard deviation. Do not claim that Group DRO improves generalization until the real runs support that conclusion.
